# eCommerce User Behavior Analysis

## Objective
Analyze user behavior and conversion funnel using clickstream data.

## Dataset
Multi-category eCommerce behavior dataset (Kaggle)

## Key Questions
- What is the conversion funnel (view → cart → purchase)?
- Where do users drop off?
- What distinguishes purchasing sessions?

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/2019-Oct.csv", nrows=1_000_000)
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   event_time     1000000 non-null  str    
 1   event_type     1000000 non-null  str    
 2   product_id     1000000 non-null  int64  
 3   category_id    1000000 non-null  int64  
 4   category_code  681869 non-null   str    
 5   brand          852440 non-null   str    
 6   price          1000000 non-null  float64
 7   user_id        1000000 non-null  int64  
 8   user_session   1000000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 68.7 MB


### Data Overview
- Dataset contains event-level data
- Key fields: event_type, user_session, price

In [4]:
df['event_time'] = pd.to_datetime(df['event_time'])

In [5]:
df['event_type'].value_counts(normalize=True)

event_type
view        0.968513
purchase    0.016848
cart        0.014639
Name: proportion, dtype: float64

### Initial Observation

The majority of user interactions are product views, while only a small fraction progress to cart or purchase, indicating a large drop-off early in the user journey.

In [6]:
print("Unique users:", df['user_id'].nunique())
print("Unique sessions:", df['user_session'].nunique())

Unique users: 163024
Unique sessions: 226374


In [7]:
session_df = df.groupby('user_session').agg({
    'event_type': list,
    'price': 'sum'
}).reset_index()

session_df.head()

,user_session,event_type,price
0,00012d23-c857-40af-b8cb-ada787bc00cc,"[view, view]",131.23
1,00016bb9-b50d-4bcb-95d1-9c375f214c66,"[view, view]",78.20
2,0001d713-f9c4-4d96-8f8c-5da2bff7bbf9,"[view, view, view, view, view, view, view, vie...",1531.86
3,000242ef-7e45-454c-a94c-6c1fd351a974,[view],251.44
4,0002c5ea-3509-4d0b-9618-7e40925005f0,"[view, purchase]",19.52


In [8]:
session_df['has_view'] = session_df['event_type'].apply(lambda x: 'view' in x)
session_df['has_cart'] = session_df['event_type'].apply(lambda x: 'cart' in x)
session_df['has_purchase'] = session_df['event_type'].apply(lambda x: 'purchase' in x)

In [ ]:
total_sessions = len(session_df)

view_sessions = session_df['has_view'].sum()
cart_sessions = session_df['has_cart'].sum()
purchase_sessions = session_df['has_purchase'].sum()

print("View → Cart:", cart_sessions / total_sessions)

View → Cart: 0.03930563289405068
Cart → Purchase: 1.609624465932089


In [14]:
cart_sessions_df = session_df[session_df['has_cart']]

In [15]:
purchase_after_cart = cart_sessions_df['has_purchase'].sum()

In [16]:
cart_to_purchase = purchase_after_cart / len(cart_sessions_df)

print("Correct Cart → Purchase:", cart_to_purchase)

Correct Cart → Purchase: 0.5586912525297953


### Funnel Insight

The conversion funnel reveals a major drop-off at the early stage.

Only ~4% of sessions proceed from viewing products to adding items to the cart, indicating low initial engagement.

Among sessions that reach the cart stage, approximately 56% convert to purchase.

This suggests that the primary bottleneck lies in encouraging users to take action (add to cart), rather than in the checkout process.

In [10]:
session_df['num_events'] = session_df['event_type'].apply(len)

session_df.groupby('has_purchase')['num_events'].mean()

has_purchase
False    4.243061
True     7.000908
Name: num_events, dtype: float64

### Engagement Insight

Sessions that result in a purchase have a significantly higher average number of interactions (~7 events) compared to non-purchasing sessions (~4 events).

This suggests that higher user engagement is strongly associated with conversion.

Users who interact more with the platform—such as viewing multiple products—are more likely to complete a purchase.

In [11]:
session_df.to_csv("../data/processed/session_level_sample.csv", index=False)

### Business Implications

- The primary bottleneck is at the product interaction stage (view → cart).
- Improving product page design, pricing strategy, or trust signals may increase add-to-cart rates.
- Since cart-to-purchase conversion is relatively high, optimizing early-stage engagement is likely to yield the highest impact on overall revenue.